In [ ]:
import sys
print("python:", sys.executable)

import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())

import transformers
print("transformers:", transformers.__version__)

# 핵심: transformers가 torch를 보고 있는지
from transformers.utils import is_torch_available
print("is_torch_available:", is_torch_available())

In [1]:
# ─── Cell 1: Import + 학습 설정 (CFG) ────────────────────────────────────────
import os
import math
import random
import numpy as np
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

from modules import *

# ── Tokenizer 경고 제거 ────────────────────────────────────────────────────────
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ── 캐시 경로 고정 (/data3/llm_download) ─────────────────────────────────────
CACHE_ROOT = "/data3/llm_download"
os.makedirs(CACHE_ROOT, exist_ok=True)
os.environ["HF_HOME"] = CACHE_ROOT
os.environ["HF_DATASETS_CACHE"] = f"{CACHE_ROOT}/datasets"

# ── 하이퍼파라미터 ─────────────────────────────────────────────────────────────
# [변경 요약 — 참고 레포(SpikingMamba) distill_smamba.yaml + multi_gpu.yaml 정렬]
#   block_size : 256  → 2048   (yaml: max_seq_length=2048)
#   batch_size : 3    → 4      (yaml: per_device_train_batch_size=4)
#   n_epochs   : 5    → 1      (yaml: num_train_epochs=1)
#   adam_betas : (0.9,0.95) → (0.9,0.999)  (HF AdamW 기본값)
#   bf16       : (없음) → True (yaml: bf16=true)
#   ce_weight  : 1.0  → 0.0    (yaml: ce_weight=0.0)
#   sgc_weight : 1.0  → 1.0    (yaml: kl_weight=1.0  ← KD를 못 쓰면 SGC만 동일 비중 유지)
#   freeze_embed: (없음) → True (distill_smamba.py: 'embedding' 들어가는 파라미터 freeze)
#   grad_accum : 1    → 8      (참고: 8 GPU × bs4 = 32, 단일 GPU 보정)
CFG = dict(
    # ── 모델 (참고 레포 기본값 기준) ────────────────────────────────────────────
    vocab_size      = 50280,   # config_mamba 기본값
    d_model         = 768,     # config_mamba 기본값
    n_layer         = 24,      # config_mamba 기본값
    d_state         = 128,     # SpikeMamba2 기본값
    expand          = 2,       # SpikeMamba2 기본값
    headdim         = 64,      # SpikeMamba2 기본값
    conv1d_kernel   = 4,
    use_gated_mlp   = False,
    sgc_indices     = None,    # None → [0, n_layer//2-1, n_layer-1]  (참고 레포와 동일)
    spike_qmin      = -4.0,
    spike_qmax      = 4.0,
    # ── 멀티 GPU 설정 ─────────────────────────────────────────────────────────
    gpu_ids         = [0],
    # ── 데이터 ────────────────────────────────────────────────────────────────
    block_size      = 256,    # [변경] 256 → 2048  (distill_smamba.yaml max_seq_length=2048)
    batch_size      = 2,       # [변경] 3 → 4       (yaml per_device_train_batch_size=4)
    num_workers     = 0,
    # ── 학습 (참고 레포 distill_smamba.yaml 기준) ─────────────────────────────
    n_epochs        = 1,       # [변경] 5 → 1       (yaml num_train_epochs=1)
    lr              = 1e-3,    # 동일 (yaml learning_rate=1.0e-03)
    warmup_ratio    = 0.01,    # 동일 (yaml warmup_ratio=0.01)
    weight_decay    = 0.0,     # 동일 (HF 기본 0.0)
    grad_clip       = 1.0,     # 동일 (HF 기본 max_grad_norm=1.0)
    adam_betas      = (0.9, 0.999),   # [변경] (0.9,0.95) → (0.9,0.999)  HF AdamW 기본값
    grad_accum_steps= 16,       # [추가] 단일 GPU에서 effective batch 32 보정 (참고: 8GPU×bs4)
    use_bf16        = True,    # [추가] yaml bf16=true 정렬 (autocast bf16)
    freeze_embed    = False,    # [추가] distill_smamba.py 'embedding' freeze 정렬
    # ── Loss 가중치 ───────────────────────────────────────────────────────────
    ce_weight       = 0.0,     # [변경] 1.0 → 0.0   (yaml ce_weight=0.0)
                               #   ※ 단, 참고 레포는 KD라 teacher의 KL이 메인 손실.
                               #     이 노트북엔 teacher가 없으므로 ce_weight=0이면 학습이 안 됨.
                               #     → 아래 use_ce_fallback 로 자동 보정 가능.
    sgc_weight      = 1.0,     # 동일 (yaml kl_weight=1.0  ← KD KL 자리에 SGC 사용)
    use_ce_fallback = True,    # [추가] teacher 없을 때 ce_weight=0이면 1.0으로 끌어올림
    # ── 기타 ──────────────────────────────────────────────────────────────────
    device          = "auto",
    seed            = 42,
    log_interval    = 10,
    save_dir        = "./ckpt_smamba",
)

# [추가] CE fallback: teacher 없는 환경에서 ce_weight=0이면 학습 신호가 사라지므로
#       자동으로 1.0으로 변경 (참고 레포 그대로 0.0 두고 싶으면 use_ce_fallback=False)
if CFG["use_ce_fallback"] and CFG["ce_weight"] == 0.0:
    print("[CFG] teacher 없음 → ce_weight 0.0 → 1.0 으로 자동 보정")
    CFG["ce_weight"] = 1.0

if CFG["device"] == "auto":
    if torch.cuda.is_available() and torch.cuda.device_count() > max(CFG["gpu_ids"]):
        CFG["device"] = f"cuda:{CFG['gpu_ids'][0]}"
    else:
        CFG["device"] = "cuda" if torch.cuda.is_available() else "cpu"

def set_seed(seed):
    random.seed(seed)                          # Python random 시드 고정
    np.random.seed(seed)                       # NumPy 시드 고정
    torch.manual_seed(seed)                    # PyTorch CPU 시드 고정
    torch.cuda.manual_seed(seed)               # PyTorch GPU 시드 고정
    torch.cuda.manual_seed_all(seed)           # PyTorch 멀티 GPU 시드 고정
    torch.backends.cudnn.deterministic = True  # 연산의 결정론적 동작 보장
    # torch.backends.cudnn.benchmark = False     # 성능 최적화 비활성화 (결정론적 보장)
    # torch.use_deterministic_algorithms(True)
    # torch.set_default_dtype(torch.float32)
    # torch.backends.cudnn.allow_tf32 = False  # TF32 연산 비활성화
    # torch.backends.cuda.matmul.allow_tf32 = False  # matmul 연산에서도 TF32 사용 금지

set_seed(CFG["seed"])
print(f"Device : {CFG['device']}")
if torch.cuda.is_available():
    print(f"Visible CUDA count={torch.cuda.device_count()}")
    print(f"Requested GPU IDs={CFG['gpu_ids']}")
print(f"HF_HOME={os.environ['HF_HOME']}")
print(f"HF_DATASETS_CACHE={os.environ['HF_DATASETS_CACHE']}")
print(f"d_model={CFG['d_model']}  n_layer={CFG['n_layer']}  "
      f"d_inner={CFG['d_model']*CFG['expand']}  "
      f"nheads={CFG['d_model']*CFG['expand']//CFG['headdim']}")


[CFG] teacher 없음 → ce_weight 0.0 → 1.0 으로 자동 보정
Device : cuda:0
Visible CUDA count=1
Requested GPU IDs=[0]
HF_HOME=/data3/llm_download
HF_DATASETS_CACHE=/data3/llm_download/datasets
d_model=768  n_layer=24  d_inner=1536  nheads=24


In [2]:
# ─── Cell 2: Tokenizer + Dataset + DataLoader ────────────────────────────────
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

# ── Tokenizer ─────────────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(
    "EleutherAI/gpt-neox-20b",
    cache_dir=os.environ["HF_HOME"],
)
# [변경] 참고 레포 distill_smamba.py: tokenizer.eos_token = "<|endoftext|>" 명시
#         이후 pad_token = eos_token 도 동일하게 지정
tokenizer.eos_token = "<|endoftext|>"
tokenizer.pad_token = tokenizer.eos_token
CFG["vocab_size"] = len(tokenizer)   # 50280
print(f"Tokenizer: EleutherAI/gpt-neox-20b  vocab={len(tokenizer)}")

# ── Dataset: wikitext-2 ───────────────────────────────────────────────────────
raw = load_dataset(
    "wikitext",
    "wikitext-2-raw-v1",
    cache_dir=os.environ["HF_DATASETS_CACHE"],
)

def tokenize_and_chunk(split, block_size):
    """전체 텍스트를 이어 붙인 뒤 (block_size+1) 크기 청크로 분할"""
    text = "\n\n".join(t for t in raw[split]["text"] if t.strip())
    ids  = tokenizer.encode(text)
    chunks = []
    for i in range(0, len(ids) - block_size, block_size):
        chunks.append(ids[i : i + block_size + 1])
    return chunks

class LMDataset(Dataset):
    def __init__(self, chunks):
        self.data = [torch.tensor(c, dtype=torch.long) for c in chunks]
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        seq = self.data[idx]
        return seq[:-1], seq[1:]   # input, label  (각각 block_size 길이)

block_size = CFG["block_size"]
train_chunks = tokenize_and_chunk("train",      block_size)
val_chunks   = tokenize_and_chunk("validation", block_size)

train_ds = LMDataset(train_chunks)
val_ds   = LMDataset(val_chunks)

train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"],
                          shuffle=True,  pin_memory=True, num_workers=CFG["num_workers"])
val_loader   = DataLoader(val_ds,   batch_size=CFG["batch_size"],
                          shuffle=False, pin_memory=True, num_workers=CFG["num_workers"])

print(f"Train: {len(train_ds):,} chunks  |  Val: {len(val_ds):,} chunks")

print(f"Train steps per epoch: {len(train_loader)}")
print(f"DataLoader num_workers={CFG['num_workers']}")

Tokenizer: EleutherAI/gpt-neox-20b  vocab=50277
Train: 9,499 chunks  |  Val: 986 chunks
Train steps per epoch: 4750
DataLoader num_workers=0


In [ ]:
# ─── Cell 3: 모델 인스턴스화 (+ optional DataParallel) ───────────────────────
base_model = SpikingMamba(
    vocab_size    = CFG["vocab_size"],
    d_model       = CFG["d_model"],
    n_layer       = CFG["n_layer"],
    d_state       = CFG["d_state"],
    expand        = CFG["expand"],
    conv1d_kernel = CFG["conv1d_kernel"],
    headdim       = CFG["headdim"],
    mlp_ratio     = 2,
    use_gated_mlp = CFG["use_gated_mlp"],
    spike_qmin    = CFG["spike_qmin"],
    spike_qmax    = CFG["spike_qmax"],
    proj_bias     = False,   # 레포와 동일
    sgc_indices   = CFG["sgc_indices"],
)

use_dp = (
    torch.cuda.is_available()
    and CFG["device"].startswith("cuda:")
    and len(CFG["gpu_ids"]) > 1
    and torch.cuda.device_count() > max(CFG["gpu_ids"]) 
)
if use_dp:
    primary = CFG["gpu_ids"][0]
    base_model = base_model.to(f"cuda:{primary}")
    model = torch.nn.DataParallel(
        base_model,
        device_ids=CFG["gpu_ids"],
        output_device=primary,
    )
else:
    model = base_model.to(CFG["device"])

model_core = model.module if isinstance(model, torch.nn.DataParallel) else model

# [추가] 참고 레포 distill_smamba.py 정렬:
#   for name, params in model.named_parameters():
#       if 'embedding' in name: params.requires_grad = False
#   → 이 노트북의 embed 파라미터 이름은 'embed.weight'. tie weight 라
#     embed.weight 를 freeze 하면 lm_head.weight 도 같이 고정됨.
if CFG.get("freeze_embed", False):
    frozen = []
    for name, p in model_core.named_parameters():
        if "embed" in name:
            p.requires_grad = False
            frozen.append(name)
    print(f"[Freeze] embedding-related params frozen: {frozen}")

total_p = sum(p.numel() for p in model.parameters())
train_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"총 파라미터: {total_p:,}  (학습 가능: {train_p:,})")

sgc_idxs = [i for i, blk in enumerate(model_core.blocks) if blk.sgc_on]
print(f"SGC on 블록: {sgc_idxs}")
print(f"DataParallel={isinstance(model, torch.nn.DataParallel)}")
if isinstance(model, torch.nn.DataParallel):
    print(f"GPU IDs={CFG['gpu_ids']}  primary=cuda:{CFG['gpu_ids'][0]}")
print(f"d_inner={CFG['d_model']*CFG['expand']}  nheads={CFG['d_model']*CFG['expand']//CFG['headdim']}")


print("모델")
print(model)

총 파라미터: 140,243,136  (학습 가능: 140,243,136)
SGC on 블록: [0, 11, 23]
DataParallel=False
d_inner=1536  nheads=24


In [ ]:
# ─── Cell 5: SGC Loss + Optimizer + Scheduler ───────────────────────────────

def sgc_alignment_loss(sgc_outputs):
    """
    smooth-spike mutual alignment loss.
    레포 mamba_kd_trainer.py 공식:
      L = ||softmax(smooth).detach - softmax(spike)||_2
        + ||softmax(smooth) - softmax(spike).detach||_2
    → in_proj 출력과 out_proj 출력 각각 적용 후 SGC 블록 수로 나눔.

    sgc_outputs: list of (proj_spike, proj_smooth, y_spike, y_smooth)
      proj_*: (B, L, 2E+2N+H)  — in_proj 출력
      y_*:    (B, L, D)         — out_proj 출력 (residual 합산 전)
    """
    total = 0.0
    for proj_s, proj_f, y_s, y_f in sgc_outputs:
        # in_proj 정렬
        sp_f = F.softmax(proj_f, dim=-1)
        sp_s = F.softmax(proj_s, dim=-1)
        total += torch.norm(sp_f.detach() - sp_s, p=2, dim=-1).mean()
        total += torch.norm(sp_f - sp_s.detach(), p=2, dim=-1).mean()
        # out_proj 정렬
        so_f = F.softmax(y_f, dim=-1)
        so_s = F.softmax(y_s, dim=-1)
        total += torch.norm(so_f.detach() - so_s, p=2, dim=-1).mean()
        total += torch.norm(so_f - so_s.detach(), p=2, dim=-1).mean()
    # 레포: hidden_proj_loss / 3  (SGC 블록 기본 3개 기준)
    return total / max(len(sgc_outputs), 1)


# 임베딩 포함 전체 파라미터 학습
trainable = [p for p in model.parameters() if p.requires_grad]
print(f"\n학습 가능 파라미터: {sum(p.numel() for p in trainable):,}")

optimizer = torch.optim.AdamW(
    trainable,
    lr=CFG["lr"],
    weight_decay=CFG["weight_decay"],
    betas=CFG["adam_betas"],   # [변경] (0.9, 0.95) → CFG["adam_betas"]=(0.9, 0.999)  HF 기본값
)

# [변경] grad_accum_steps 반영하여 optimizer step 횟수 기준으로 스케줄러 길이 산정
#   참고 레포: scheduler step은 optimizer.step() 호출마다 발생 → micro-step이 아니라 update step.
updates_per_epoch = math.ceil(len(train_loader) / CFG["grad_accum_steps"])
total_steps  = updates_per_epoch * CFG["n_epochs"]
warmup_steps = int(total_steps * CFG["warmup_ratio"])

from transformers import get_cosine_schedule_with_warmup
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

# ── 스케줄러 설정 요약 출력 ────────────────────────────────────────────────────
# updates_per_epoch : 1 에포크에서 실제로 optimizer.step()이 몇 번 호출되는지
#   = ceil(배치 수 / grad_accum_steps)
#   예) 배치 1000개, grad_accum=16  →  ceil(1000/16) = 63 update/epoch
# total_steps       : 전체 학습 동안의 optimizer update 횟수 = updates_per_epoch × n_epochs
# warmup_steps      : lr을 0 → lr_peak 까지 선형으로 올리는 구간 (total_steps × warmup_ratio)
# lr_peak           : warmup 이후 코사인 감소의 최고점 (= CFG["lr"])
print(f"배치 수(train_loader): {len(train_loader)}  |  grad_accum_steps: {CFG['grad_accum_steps']}")
print(f"  → updates_per_epoch : {updates_per_epoch}  (= ceil({len(train_loader)} / {CFG['grad_accum_steps']}))")
print(f"  → total_steps       : {total_steps}  (= {updates_per_epoch} × {CFG['n_epochs']} epoch)")
print(f"  → warmup_steps      : {warmup_steps}  (= int({total_steps} × {CFG['warmup_ratio']}))")
print(f"  → lr_peak           : {CFG['lr']:.1e}")



학습 가능 파라미터: 140,243,136
총 스텝: 297  |  warmup: 2  |  lr_peak: 1.0e-03


In [ ]:
import time
from contextlib import nullcontext
os.makedirs(CFG["save_dir"], exist_ok=True)

# [추가] bf16 autocast 컨텍스트 (참고 yaml bf16=true)
def _amp_ctx():
    if CFG.get("use_bf16", False) and torch.cuda.is_available():
        return torch.autocast(device_type="cuda", dtype=torch.bfloat16)
    return nullcontext()

def safe_ppl(loss_value, max_exp=80.0):
    """exp overflow/포화 방지용 PPL 계산"""
    if loss_value >= max_exp:
        return float("inf")
    return math.exp(loss_value)

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_loss, total_n = 0.0, 0
    eval_start_time = time.time()
    total_eval_steps = len(loader)
    eval_log_interval = max(1, total_eval_steps // 10)  # 10번 정도 진행상황 출력

    print(f"  [Eval] started: total_steps={total_eval_steps}")
    for eval_step, (x, y) in enumerate(loader, start=1):
        x, y = x.to(device), y.to(device)
        with _amp_ctx():                                    # [추가] eval에도 bf16
            logits, _ = model(x)
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), y.reshape(-1))
        total_loss += loss.item() * x.size(0)
        total_n += x.size(0)

        if eval_step % eval_log_interval == 0 or eval_step == total_eval_steps:
            elapsed = time.time() - eval_start_time
            print(f"  [Eval] step {eval_step}/{total_eval_steps} | elapsed={elapsed//60:.0f}m{elapsed%60:.0f}s")

    avg_loss = total_loss / total_n
    return avg_loss, safe_ppl(avg_loss)


In [ ]:
# ─── Cell 5: Training Loop ───────────────────────────────────────────────────
# [변경 요약]
#   - bf16 autocast 추가  (CFG["use_bf16"], 참고 yaml bf16=true 정렬)
#   - gradient accumulation 추가 (CFG["grad_accum_steps"], 단일 GPU 보정)
#   - scheduler.step() 은 optimizer.step() 호출과 동기화 (참고 레포 HF Trainer 동작)

device      = CFG["device"]
best_val    = float("inf")
step        = 0                       # micro-step (배치 단위)
update_step = 0                       # [추가] optimizer update step 카운터
accum_steps = CFG["grad_accum_steps"] # [추가]
total_train_steps = updates_per_epoch * CFG["n_epochs"]
epoch_start_time = None

print(f"[Train] grad_accum_steps={accum_steps}  use_bf16={CFG['use_bf16']}  "
      f"updates/epoch={updates_per_epoch}  total_updates={total_train_steps}")

for epoch in range(CFG["n_epochs"]):
    epoch_start_time = time.time()
    model.train()
    run_ce = run_sgc = run_tot = 0.0

    print(f"\n{'='*70}")
    print(f"Epoch {epoch+1}/{CFG['n_epochs']} started...")
    print(f"{'='*70}")

    optimizer.zero_grad()             # [변경] 매 micro-step이 아니라 accum 시작 시 zero
    for batch_idx, (x, y) in enumerate(train_loader):
        x, y = x.to(device), y.to(device)   # (B, L)

        with _amp_ctx():                                    # [추가] bf16 autocast
            logits, sgc_out = model(x)                      # logits: (B,L,vocab)

            # 첫 스텝에서 로그잇 스케일 점검
            if epoch == 0 and batch_idx == 0:
                print(
                    f"[Debug] logits mean={logits.mean().item():.4f} std={logits.std().item():.4f} "
                    f"min={logits.min().item():.4f} max={logits.max().item():.4f}"
                )

            # CE loss
            loss_ce = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                y.reshape(-1),
            )

            # SGC loss (smooth-spike 정렬, 레포 mamba_kd_trainer 공식)
            if sgc_out:
                loss_sgc = sgc_alignment_loss(sgc_out)
            else:
                loss_sgc = torch.tensor(0.0, device=device)

            loss = CFG["ce_weight"] * loss_ce + CFG["sgc_weight"] * loss_sgc

        # [변경] grad accumulation: loss 를 accum_steps 로 나눠 micro-step마다 backward
        (loss / accum_steps).backward()

        run_ce  += loss_ce.item()
        run_sgc += loss_sgc.item()
        run_tot += loss.item()
        step    += 1

        # [변경] accum_steps 마다 optimizer.step / scheduler.step (참고 레포 동작과 동기)
        is_last_micro = ((batch_idx + 1) == len(train_loader))
        if (step % accum_steps == 0) or is_last_micro:
            torch.nn.utils.clip_grad_norm_(
                (p for p in model.parameters() if p.requires_grad),
                CFG["grad_clip"],
            )
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            update_step += 1

        if (batch_idx + 1) % CFG["log_interval"] == 0:
            n = CFG["log_interval"]
            elapsed = time.time() - epoch_start_time
            avg_ce = run_ce / n
            avg_ppl = safe_ppl(avg_ce)
            avg_bpc = avg_ce / math.log(2)
            lr_now = scheduler.get_last_lr()[0]
            ppl_str = f"{avg_ppl:.3e}" if math.isfinite(avg_ppl) else "inf(>exp(80))"

            print(f"[Epoch {epoch+1}] Step {batch_idx+1}/{len(train_loader)} "
                  f"(upd={update_step}/{total_train_steps}) | "
                  f"CE={avg_ce:.4f} | SGC={run_sgc/n:.4f} | PPL={ppl_str} | BPC={avg_bpc:.2f} | "
                  f"LR={lr_now:.2e} | Time={elapsed//60:.0f}m{elapsed%60:.0f}s")
            run_ce = run_sgc = run_tot = 0.0

    # ── Validation ───────────────────────────────────────────────────────────
    epoch_elapsed = time.time() - epoch_start_time
    print(f"\n[Epoch {epoch+1}] Training done ({epoch_elapsed//60:.0f}m{epoch_elapsed%60:.0f}s)")
    print(f"  Running validation...")

    val_loss, val_ppl = evaluate(model, val_loader, device)
    val_ppl_str = f"{val_ppl:.3e}" if math.isfinite(val_ppl) else "inf(>exp(80))"
    print(f"  Val Loss={val_loss:.4f}  Val PPL={val_ppl_str}")

    if val_loss < best_val:
        best_val  = val_loss
        ckpt_path = os.path.join(CFG["save_dir"], "best.pt")
        model_to_save = model.module if isinstance(model, torch.nn.DataParallel) else model
        torch.save({"epoch": epoch + 1, "model": model_to_save.state_dict(),
                    "cfg": CFG, "val_loss": val_loss}, ckpt_path)
        print(f"  ✓ Checkpoint saved: {ckpt_path}")

print(f"\n{'='*70}")
best_ppl = safe_ppl(best_val)
best_ppl_str = f"{best_ppl:.3e}" if math.isfinite(best_ppl) else "inf(>exp(80))"
print(f"학습 완료!  best val_loss={best_val:.4f}  ppl={best_ppl_str}")
print(f"{'='*70}")

[Train] grad_accum_steps=16  use_bf16=True  updates/epoch=297  total_updates=297

Epoch 1/1 started...
[Debug] logits mean=-0.0056 std=27.7500 min=-146.0000 max=348.0000
[Epoch 1] Step 10/4750 (upd=0/297) | CE=276.1424 | SGC=0.2052 | PPL=inf(>exp(80)) | BPC=398.39 | LR=0.00e+00 | Time=0m53s
[Epoch 1] Step 20/4750 (upd=1/297) | CE=275.6376 | SGC=0.2050 | PPL=inf(>exp(80)) | BPC=397.66 | LR=5.00e-04 | Time=1m45s
[Epoch 1] Step 30/4750 (upd=1/297) | CE=276.9330 | SGC=0.2052 | PPL=inf(>exp(80)) | BPC=399.53 | LR=5.00e-04 | Time=2m41s
[Epoch 1] Step 40/4750 (upd=2/297) | CE=196.8716 | SGC=0.1966 | PPL=inf(>exp(80)) | BPC=284.03 | LR=1.00e-03 | Time=3m35s
[Epoch 1] Step 50/4750 (upd=3/297) | CE=162.0946 | SGC=0.1869 | PPL=inf(>exp(80)) | BPC=233.85 | LR=1.00e-03 | Time=4m30s
[Epoch 1] Step 60/4750 (upd=3/297) | CE=112.8176 | SGC=0.1571 | PPL=inf(>exp(80)) | BPC=162.76 | LR=1.00e-03 | Time=5m22s
[Epoch 1] Step 70/4750 (upd=4/297) | CE=112.0393 | SGC=0.1432 | PPL=inf(>exp(80)) | BPC=161.64 | L

In [ ]:
# ─── Cell 7: Greedy / Top-k Generation ───────────────────────────────────────
@torch.no_grad()
def generate(model, prompt, max_new_tokens=200, temperature=0.7, top_k=40):
    """Greedy(top_k=1) 또는 top-k 샘플링으로 텍스트 생성"""
    device = CFG["device"]
    model.eval()
    ids = torch.tensor(tokenizer.encode(prompt), dtype=torch.long).unsqueeze(0).to(device)
    for _ in range(max_new_tokens):
        logits, _ = model(ids)
        logits = logits[:, -1, :] / temperature        # (1, vocab)
        if top_k > 1:
            v, _ = torch.topk(logits, top_k)
            logits[logits < v[:, [-1]]] = float("-inf")
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
        else:
            next_id = logits.argmax(dim=-1, keepdim=True)
        ids = torch.cat([ids, next_id], dim=-1)
        if next_id.item() == tokenizer.eos_token_id:
            break
    return tokenizer.decode(ids[0].tolist(), skip_special_tokens=True)

# ── 실행 ───────────────────────────────────────────────────────────────────────
val_loss, val_ppl = evaluate(model, val_loader, CFG["device"])
print(f"최종  val_loss={val_loss:.4f}  PPL={val_ppl:.2f}")

for prompt in [
    "The future of artificial intelligence is",
    "Once upon a time in a land far away",
]:
    print(f"\n[Prompt]    {prompt}")
    print(f"[Generated] {generate(model, prompt)}")


  [Eval] started: total_steps=329
  [Eval] step 32/329 | elapsed=3m3s
  [Eval] step 64/329 | elapsed=6m8s
  [Eval] step 96/329 | elapsed=9m12s
  [Eval] step 128/329 | elapsed=12m11s
  [Eval] step 160/329 | elapsed=15m16s
  [Eval] step 192/329 | elapsed=18m22s
  [Eval] step 224/329 | elapsed=21m27s
  [Eval] step 256/329 | elapsed=24m32s
  [Eval] step 288/329 | elapsed=27m41s
  [Eval] step 320/329 | elapsed=30m48s
  [Eval] step 329/329 | elapsed=31m36s
최종  val_loss=5.8994  PPL=364.82

[Prompt]    The future of artificial intelligence is
[Generated] The future of artificial intelligence is still an active , but when it peaked at 5 @.@ 76 in ( 17 @.@ 5 mm ) at low on the Pistons . In one part of the road , the first in which point is considered to be a common names and is , a total of about 0 @.

[Prompt]    Once upon a time in a land far away
[Generated] Once upon a time in a land far away from the title , including a second three @-@ one on the west side . The total total , which was lat

In [ ]:
# ─── Debug Cell: One-batch loss/logit scale check ─────────────────────────────
with torch.no_grad():
    model.eval()
    x_dbg, y_dbg = next(iter(train_loader))
    x_dbg, y_dbg = x_dbg.to(CFG["device"]), y_dbg.to(CFG["device"])
    logits_dbg, _ = model(x_dbg)
    ce_dbg = F.cross_entropy(logits_dbg.reshape(-1, logits_dbg.size(-1)), y_dbg.reshape(-1))
    print(f"[Debug] one-batch CE={ce_dbg.item():.4f}")
    print(
        f"[Debug] logits mean={logits_dbg.mean().item():.4f} std={logits_dbg.std().item():.4f} "
        f"min={logits_dbg.min().item():.4f} max={logits_dbg.max().item():.4f}"
    )
    print(f"[Debug] current LR={scheduler.get_last_lr()[0]:.2e}")

[Debug] one-batch CE=4.3122
[Debug] logits mean=-9.9059 std=6.4490 min=-41.3688 max=40.8616
[Debug] current LR=0.00e+00
